<a href="https://colab.research.google.com/github/catorrecampo-sys/-GE-120-1A-Final-Project-Group-5/blob/main/Leveling_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [96]:
# CHECK / IMPORT PANDAS
try:
    import pandas as pd
    print('Pandas imported successfully.')

except ImportError:
    print('Error! Pandas is not installed.')
    print('Install using: pip install pandas') # for context, I did not use force %pip install pandas since we're not sure if they're using jupyter
    exit()

Pandas imported successfully.


In [97]:

class FileHandler:
    '''
    Handles:
        - file validation
        - csv reading
        - leveling type detection
    '''

    def __init__(self):
        self.extension = '.csv'
        self.three_wire_columns = [
            'BS_point',
            'BS_upper',
            'BS_middle',
            'BS_lower',
            'FS_point',
            'FS_upper',
            'FS_middle',
            'FS_lower'
        ]
        self.differential_columns = [
            'BS_point',
            'BS_Reading',
            'BS_Dist',
            'FS_point',
            'FS_Readings',
            'FS_Dist'
        ]
        self.filename = ''
        self.leveling_data = None
        self.leveling_type = ''

    def read_file(self, file_path_or_url=None):
        '''
        '''

        while True:
            if file_path_or_url:
                self.filename = file_path_or_url
            else:
                self.filename = input('Enter name of csv file to open: ')

            # Automatically add .csv
            if not self.filename.endswith(self.extension) and not self.filename.startswith('http'):
                self.filename += self.extension

            try:
                self.leveling_data = pd.read_csv(self.filename)

            except FileNotFoundError:
                print('Error! File not found.')
                if file_path_or_url: break
                continue

            except pd.errors.EmptyDataError:
                print('Error! CSV file is empty.')
                if file_path_or_url: break
                continue

            except pd.errors.ParserError:
                print('Error! Invalid CSV formatting.')
                if file_path_or_url: break
                continue

            except Exception as e:
                print(f'Error! Could not read file: {e}')
                if file_path_or_url: break
                continue

            # Detect leveling type
            self.detect_leveling_type()

            # If valid format
            if self.leveling_type != '':
                break
            elif file_path_or_url:
                break

        return self.leveling_data, self.leveling_type

    def detect_leveling_type(self):
        '''
        '''

        columns = self.leveling_data.columns.tolist()

        # 3-WIRE
        if all(col in columns for col in self.three_wire_columns):
            self.leveling_type = '3-Wire Leveling'

            print(f'\nOpening {self.filename}')
            print('Detected: 3-Wire Leveling Data')

        # DIFFERENTIAL
        elif all(col in columns for col in self.differential_columns):
            self.leveling_type = 'Differential Leveling'

            print(f'\nOpening {self.filename}')
            print('Detected: Differential Leveling Data')

        # INVALID
        else:
            self.leveling_type = ''

            print('Error! CSV format is not recognized.')
            print('Check column names and file structure.')

# Create FileHandler object to manage CSV input and detection
handler = FileHandler()

In [98]:

# Store leveling data file
leveling_data = "https://raw.githubusercontent.com/catorrecampo-sys/-GE-120-1A-Final-Project-Group-5/main/leveling_data.csv"

# Read CSV file and get dataset + leveling type
# Read the file into a variable
my_leveling_data, level_type1 = handler.read_file(leveling_data)

# Print it to check if it works
if my_leveling_data is not None:
    print("\nLeveling Data:")
    print(my_leveling_data.head())

# Store 3-wire leveling data file
threewire_data = "https://raw.githubusercontent.com/catorrecampo-sys/-GE-120-1A-Final-Project-Group-5/main/leveling3wire_data.csv"

# Read the file into a variable
threewire_data, level_type2 = handler.read_file(threewire_data)

# print it to check if it works
if threewire_data is not None:
    print("\nThree-Wire Leveling Data:")
    print(threewire_data.head())


Opening https://raw.githubusercontent.com/catorrecampo-sys/-GE-120-1A-Final-Project-Group-5/main/leveling_data.csv
Detected: Differential Leveling Data

Leveling Data:
  BS_point  BS_Reading  BS_Dist FS_point  FS_Readings  FS_Dist
0      BM1        0.70       38     TP 1        1.990       38
1     TP 1        1.53       26     TP 2        0.930       26
2     TP 2        1.57       15     TP 3        1.000       11
3     TP 3        1.66       16     TP 4        1.020       16
4     TP 4        1.82       18     TP 5        0.836       19

Opening https://raw.githubusercontent.com/catorrecampo-sys/-GE-120-1A-Final-Project-Group-5/main/leveling3wire_data.csv
Detected: 3-Wire Leveling Data

Three-Wire Leveling Data:
  BS_point  BS_upper  BS_middle  BS_lower FS_point  FS_upper  FS_middle  \
0      BM1      0.89       0.70      0.51     TP 1      2.18       1.99   
1     TP 1      1.66       1.53      1.40     TP 2      1.06       0.93   
2     TP 2      1.65       1.57      1.50     TP 

Differential

In [99]:
print("\nStarting Differential Leveling...")

# assumed benchmark elevation, idk kung input ba dapat to or somehting
starting_elev = 100.0
running_elev = starting_elev

# storing values here first before pushing to dataframe
HI_values = []
elevation_values = []

# using iterrows for readability
for i, dataRow in my_leveling_data.iterrows():

    bs_reading = dataRow['BS_Reading']
    fs_reading = dataRow['FS_Readings']

    # Height of Instrument formula
    current_HI = running_elev + bs_reading

    HI_values.append(current_HI)

    # compute next elevation
    next_elev = current_HI - fs_reading

    elevation_values.append(next_elev)

    # update running elevation for next loop
    running_elev = next_elev

# add newly computed columns
my_leveling_data['HI'] = HI_values
my_leveling_data['Elevation'] = elevation_values

print("\n--- Differential Leveling Table (Unadjusted) ---")

cols_to_show = [
    'BS_point',
    'BS_Reading',
    'HI',
    'FS_point',
    'FS_Readings',
    'Elevation'
]

print(my_leveling_data[cols_to_show])

# misclosure

total_bs = my_leveling_data['BS_Reading'].sum()
total_fs = my_leveling_data['FS_Readings'].sum()

# keeping this rounded because long decimals are annoying to read
closure_error = round(total_bs - total_fs, 3)

print("\n--- Differential Leveling Misclosure ---")
print(f"Starting Elevation : {starting_elev}")
print(f"Total BS Readings  : {round(total_bs, 3)}")
print(f"Total FS Readings  : {round(total_fs, 3)}")
print(f"Misclosure Error   : {closure_error}")


Starting Differential Leveling...

--- Differential Leveling Table (Unadjusted) ---
   BS_point  BS_Reading       HI FS_point  FS_Readings  Elevation
0       BM1       0.700  100.700     TP 1        1.990     98.710
1      TP 1       1.530  100.240     TP 2        0.930     99.310
2      TP 2       1.570  100.880     TP 3        1.000     99.880
3      TP 3       1.660  101.540     TP 4        1.020    100.520
4      TP 4       1.820  102.340     TP 5        0.836    101.504
5      TP 5       1.750  103.254     TP 6        0.726    102.528
6      TP 6       1.790  104.318     TP 7        0.790    103.528
7      TP 7       1.910  105.438     TP 8        0.550    104.888
8      TP 8       2.210  107.098     TP 9        0.756    106.342
9      TP 9       1.010  107.352    TP 10        1.820    105.532
10    TP 10       0.610  106.142    TP 11        1.650    104.492
11    TP 11       0.700  105.192    TP 12        1.730    103.462
12    TP 12       0.710  104.172    TP 13        1.040   

3mar

In [100]:
print("\nStarting 3-Wire Leveling...")

# assumed elevation for first point, again idk if neet ba input or something
base_elev_3wire = 100.0
current_elev_3wire = base_elev_3wire

HI_list_3wire = []
elev_list_3wire = []

for idx, row in threewire_data.iterrows():

    # average BS wire readings
    bs_avg = (
        row['BS_upper'] +
        row['BS_middle'] +
        row['BS_lower']
    ) / 3

    # average FS wire readings
    fs_avg = (
        row['FS_upper'] +
        row['FS_middle'] +
        row['FS_lower']
    ) / 3

    # compute HI
    hi_value = current_elev_3wire + bs_avg
    HI_list_3wire.append(hi_value)

    # compute new elevation
    updated_elev = hi_value - fs_avg
    elev_list_3wire.append(updated_elev)

    current_elev_3wire = updated_elev

# save results back into dataframe
threewire_data['HI'] = HI_list_3wire
threewire_data['Elevation'] = elev_list_3wire

# storing means separately because we might use them later
threewire_data['BS_mean'] = (
    threewire_data['BS_upper'] +
    threewire_data['BS_middle'] +
    threewire_data['BS_lower']
) / 3

threewire_data['FS_mean'] = (
    threewire_data['FS_upper'] +
    threewire_data['FS_middle'] +
    threewire_data['FS_lower']
) / 3

print("\n--- 3-Wire Leveling Table (Unadjusted) ---")

# showing averages only to avoid cluttering terminal output
print(
    threewire_data[
        [
            'BS_point',
            'BS_mean',
            'HI',
            'FS_point',
            'FS_mean',
            'Elevation'
        ]
    ]
)

# misclosure checking
total_bs_3wire = threewire_data['BS_mean'].sum()
total_fs_3wire = threewire_data['FS_mean'].sum()

misclosure_3wire = round(total_bs_3wire - total_fs_3wire, 3)

print("\n--- 3-Wire Misclosure Report ---")
print(f"Starting Elevation : {base_elev_3wire}")
print(f"Total BS Means     : {round(total_bs_3wire, 3)}")
print(f"Total FS Means     : {round(total_fs_3wire, 3)}")
print(f"Misclosure Error   : {misclosure_3wire}")

# TODO:
# add adjustment
# yung errror classification
# anong UI aaaaaaaa


Starting 3-Wire Leveling...

--- 3-Wire Leveling Table (Unadjusted) ---
   BS_point   BS_mean          HI FS_point   FS_mean   Elevation
0       BM1  0.700000  100.700000     TP 1  1.990000   98.710000
1      TP 1  1.530000  100.240000     TP 2  0.930000   99.310000
2      TP 2  1.573333  100.883333     TP 3  1.003333   99.880000
3      TP 3  1.660000  101.540000     TP 4  1.020000  100.520000
4      TP 4  1.820000  102.340000     TP 5  0.836667  101.503333
5      TP 5  1.750000  103.253333     TP 6  0.726667  102.526667
6      TP 6  1.790000  104.316667     TP 7  0.790000  103.526667
7      TP 7  1.910000  105.436667     TP 8  0.550000  104.886667
8      TP 8  2.210000  107.096667     TP 9  0.756667  106.340000
9      TP 9  1.010000  107.350000    TP 10  1.820000  105.530000
10    TP 10  0.610000  106.140000    TP 11  1.650000  104.490000
11    TP 11  0.700000  105.190000    TP 12  1.730000  103.460000
12    TP 12  0.710000  104.170000    TP 13  1.036667  103.133333
13    TP 13  0.80